# Stress Analysis, FEA Results & ASME B31.3 Compliance

In this notebook, we'll run through the structural evaluation of a piping system using Code_Aster result artifacts. Stress, deformation, reaction, and compliance plots are only created after real solver output tables are present.

### What you'll learn

1. How to define load cases (pressure, temperature, gravity).
2. How Code_Aster result artifacts are imported into Tuba v4.
3. How to visualize stress distributions and deformed shapes in 3D.
4. How to perform ASME B31.3 code compliance checks.
5. How stress intensification factors (SIFs) are calculated at bend and Tee junctions.

## 1. Environment Setup & Imports

In [ ]:
import sys
from pathlib import Path
import pyvista as pv

# Setup repo root for path import
REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tuba import Model
from tuba.analysis.code_aster_notebook import load_or_run_code_aster_results
from tuba.compliance.asme_b313 import ASMEB313Evaluator
from tuba.compliance.sif import compute_sifs
from tuba.visualizer import plots

# Enable interactive notebook rendering
# Defaults to zoomable 'client' locally; set TUBA_NOTEBOOK_BACKEND=static for nbconvert/CI.
from tuba.visualizer.notebook import configure_notebook_backend
JUPYTER_BACKEND = configure_notebook_backend()

## 2. Building the Piping Model

We'll build a standard L-shaped piping layout with anchors at both ends and a lateral guide restraint close to the bend.

In [ ]:
model = Model("StressDemo", standard="ASME_B31.3")

# Material P265GH (allowable stress at different temperatures in Pa)
model.add_material(
    "P265GH",
    E=2.0e11,     # Young's modulus [Pa]
    nu=0.3,       # Poisson's ratio
    rho=7850.0,   # density [kg/m3]
    alpha=1.2e-5, # thermal expansion [1/K]
    allowable_stress={20.0: 137e6, 200.0: 120e6}
)

# Section: 4-inch Schedule 40
model.add_pipe_section("4inch_sch40", OD=0.1143, WT=0.00602, corrosion_allowance=0.001)

# Routing the L-shape pipe
with model.pipe(section="4inch_sch40", material="P265GH") as b:
    b.start([0, 0, 0], support="anchor")
    b.run(5.0)
    b.bend(radius=0.2, angle=90.0, plane="XY")
    b.run(3.0)
    b.end(support="anchor")

# Add a guide support at node N2 (the exit node of the bend)
model.add_support("N2", type="guide")

# Define operating load case
model.define_load_case("Operating", gravity=True, pressure=1.5e6, temperature=200.0)

print(model)

## 3. Loading Code_Aster FEA Results

This notebook does not synthesize solver values. The next cell exports Code_Aster input files if needed, then imports existing Code_Aster output tables from the same work directory. If those tables are missing, the cell stops before any result visualization or compliance check can run.

In [ ]:
RUN_CODE_ASTER = True
CODE_ASTER_EXEC_METHOD = "wsl"  # Use "docker" if your Code_Aster installation is containerized.
CODE_ASTER_DOCKER_IMAGE = None
CODE_ASTER_WORK_DIR = REPO_ROOT / "notebooks" / "code_aster_results" / "stress_analysis_operating"

code_aster_run = load_or_run_code_aster_results(
    model,
    "Operating",
    CODE_ASTER_WORK_DIR,
    run_solver=RUN_CODE_ASTER,
    exec_method=CODE_ASTER_EXEC_METHOD,
    docker_image=CODE_ASTER_DOCKER_IMAGE,
)
results = code_aster_run.results
code_aster_artifact = code_aster_run.artifact

if code_aster_run.ran_solver:
    print("Code_Aster solver executed because result tables were missing.")
print(f"Loaded Code_Aster results from: {CODE_ASTER_WORK_DIR.resolve()}")
print(f"Node result count: {len(results.node_results)}")
print(f"Element result count: {len(results.element_results)}")

## 4. Visualization Galleries

Tuba provides high-level plotting helpers to view Code_Aster output formats directly inside the notebook.

### A. Deformed Shape
The `.plot_deformed()` method warps the geometry by displacement vectors.

In [ ]:
plots.plot_deformed(results, scale=100.0, model=model)

### B. Stress Distribution
The `.plot_stress()` method color-maps the Von Mises stress onto the inflated 3D pipe surface.

In [ ]:
plots.plot_stress(results, model=model)

### C. Combined Deformed Stress
The primary review view shows the warped shape colored by stress.

In [ ]:
plots.plot_deformed_stress(results, deform_scale=100.0, model=model)

### D. Displacement Vectors & Support Reactions
You can also display displacement vectors as arrow glyphs or reaction forces at anchor supports.

In [ ]:
# Plot displacements as arrows
plots.plot_displacement_vectors(results, scale=100.0, model=model)

# Plot support reactions
plots.plot_reactions(results, scale=1e-3, model=model)

## 5. ASME B31.3 Compliance Checking

The `ASMEB313Evaluator` checks every element end for:
1. **Sustained Stress ($S_L$):** Pressure + Weight stress must be less than hot allowable stress $S_h$.
2. **Expansion Stress ($S_E$):** Thermal expansion stress must be less than allowable expansion range $S_A = f(1.25 S_c + 0.25 S_h)$.

In [ ]:
evaluator = ASMEB313Evaluator()
report = evaluator.evaluate(model, results)

print(f"ASME B31.3 Compliance Check Results:")
print(f"  Overall Pass: {report.overall_pass}")
print(f"  Worst Sustained Stress Ratio: {report.worst_sustained_ratio:.3f}")
print(f"  Worst Expansion Stress Ratio: {report.worst_expansion_ratio:.3f}")

### Traceability: Detailed Calculations

A key feature of Tuba's compliance module is **mathematical traceability**. You can extract a step-by-step calculation trace formatted in Markdown/LaTeX for any element.

In [ ]:
from IPython.display import display, Markdown

# Get calculation trace for the straight pipe element
trace_md = report.get_detailed_calculation("pipe_str_0")
display(Markdown(trace_md[:1500] + "\n... [truncated] ..."))

## 6. SIF Computation at Tees

Stress Intensification Factors (SIFs) are used to account for stress concentrations at fittings (bends, Tees). Let's construct a branch connection model to compute the out-of-plane and in-plane SIFs.

In [ ]:
model2 = Model("TeeDemo")
model2.add_material("Steel", E=2.0e11, nu=0.3, alpha=1.2e-5, rho=7850)
model2.add_pipe_section("RunPipe", OD=0.1683, WT=0.0071)

# Create main run with midpoint node N1
with model2.pipe(section="RunPipe", material="Steel") as b:
    b.start([0, 0, 0]).run(2.0).run(2.0).end()

# Branch connection starting at N1
with model2.pipe(section="RunPipe", material="Steel") as b:
    b.start([2.0, 0.0, 0.0]).set_direction([0.0, 1.0, 0.0]).run(1.5).end()

# Define N1 as a welding Tee fitting
model2.define_tee("N1", type="welding_tee")

# Calculate SIFs for the run element connected to the Tee
run_element = model2.elements[0]
i_i, i_o, k, h = compute_sifs(run_element, model2, node_id="N1")

print(f"Welding Tee SIF results at node N1:")
print(f"  In-plane SIF (i_i): {i_i:.3f}")
print(f"  Out-of-plane SIF (i_o): {i_o:.3f}")
print(f"  Flexibility factor (k): {k:.3f}")
print(f"  Characteristic (h): {h:.3f}")

## Key Takeaways

- Code_Aster output tables are the source for all stress, deformation, and reaction displays in this notebook.
- Parsed solver results bridge Code_Aster and Tuba compliance checkers.
- `ASMEB313Evaluator` validates both sustained ($S_L$) and expansion ($S_E$) stresses.
- Detailed step-by-step markdown traces make code compliance audits extremely easy.
- Tuba automatically computes stress intensification factors at branch connections and elbows based on ASME B31.3 Appendix D.